# Building LLM-Powered Systems with PydanticAI

This notebook demonstrates how to use PydanticAI for building robust, type-safe LLM applications with structured responses and advanced features.

In [ ]:
from typing import Dict, List, Optional
import nest_asyncio
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry, RunContext, Tool
from pydantic_ai.models.openai import OpenAIModel

nest_asyncio.apply()
model = OpenAIModel("gpt-4")

## 1. Basic Agent Usage

Let's start with a simple example showing how to create and use a basic PydanticAI agent.

In [ ]:
# Create a basic agent
agent = Agent(
    model=model,
    system_prompt="You are a helpful customer support agent. Be concise and friendly.",
)

# Run a basic query
response = agent.run_sync("How can I track my order #12345?")

print("Response:", response.data)
print("\nMessage History:", response.all_messages())
print("\nCost:", response.cost())

## 2. Structured Response Agents

Now let's create an agent that returns structured, type-safe responses using Pydantic models.

In [ ]:
class ResponseModel(BaseModel):
    """Structured response with metadata."""

    response: str
    needs_escalation: bool
    follow_up_required: bool
    sentiment: str = Field(description="Customer sentiment analysis")


# Create agent with structured output
structured_agent = Agent(
    model=model,
    result_type=ResponseModel,
    system_prompt="You are an intelligent customer support agent. Analyze queries carefully and provide structured responses.",
)

response = structured_agent.run_sync("I've been waiting for my order for 2 weeks now!")
print(response.data.model_dump_json(indent=2))

## 3. Agents with Dependencies

This section shows how to work with complex data models and runtime dependencies.

In [ ]:
# Define data models
class Order(BaseModel):
    """Structure for order details."""

    order_id: str
    status: str
    items: List[str]


class CustomerDetails(BaseModel):
    """Structure for incoming customer queries."""

    customer_id: str
    name: str
    email: str
    orders: Optional[List[Order]] = None


# Create agent with dependencies
agent_with_deps = Agent(
    model=model,
    result_type=ResponseModel,
    deps_type=CustomerDetails,
    system_prompt="You are an intelligent support agent. Greet customers by name and provide personalized responses.",
)

# Example customer data
customer = CustomerDetails(
    customer_id="1",
    name="John Doe",
    email="john@example.com",
    orders=[Order(order_id="12345", status="shipped", items=["Laptop", "Mouse"])],
)

response = agent_with_deps.run_sync("What did I order?", deps=customer)
print(response.data.model_dump_json(indent=2))

## 4. Agents with Tools

Enhance agents by adding custom tools for accessing external data or performing specific actions.

In [ ]:
# Create a mock shipping database
shipping_info_db: Dict[str, str] = {
    "12345": "Shipped on 2024-12-01",
    "67890": "Out for delivery",
}


def get_shipping_info(ctx: RunContext[CustomerDetails]) -> str:
    """Tool to get shipping information."""
    return shipping_info_db[ctx.deps.orders[0].order_id]


# Create agent with tools
agent_with_tools = Agent(
    model=model,
    result_type=ResponseModel,
    deps_type=CustomerDetails,
    system_prompt="You are a support agent with access to shipping information. Use tools when needed.",
    tools=[Tool(get_shipping_info, takes_ctx=True)],
)

response = agent_with_tools.run_sync("What's the status of my order?", deps=customer)
print(response.data.model_dump_json(indent=2))

## 5. Advanced Agent Features

Demonstrate self-reflection, error handling, and automated retries.

In [ ]:
# Create an agent with retry capability
advanced_agent = Agent(
    model=model,
    result_type=ResponseModel,
    deps_type=CustomerDetails,
    retries=3,
    system_prompt="You are an intelligent agent with self-correction capabilities.",
)


@advanced_agent.tool_plain()
def get_shipping_status(order_id: str) -> str:
    """Get shipping status with error handling."""
    if not order_id.startswith("#"):
        raise ModelRetry("Order ID must start with #. Self-correcting...")

    order_id = order_id.lstrip("#")
    status = shipping_info_db.get(order_id)
    if not status:
        raise ModelRetry(f"No shipping information found for order {order_id}")
    return status


# Test the advanced agent
response = advanced_agent.run_sync("Check order 12345 status", deps=customer)
print(response.data.model_dump_json(indent=2))